In [2]:
import json
import os
import re
import sys
import pickle
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

sys.path.append('..')

from llm.pseudograph_relabelling_llm import PseudoGraphRelabellingLLM

QUEUE_FILE = r'D:\claimpkg\claimpkg-clone\src\resources\relabel_queue.pickle'
KEY_FILE = r'D:\claimpkg\claimpkg-clone\src\relabelling\keys.json'
DATA_FILE = r'D:\claimpkg\claimpkg-clone\src\resources\classified_book_dataset_6300.pickle'
OUTPUT_FILE = r'D:\claimpkg\claimpkg-clone\src\relabelling\result.json'

class RelabellingLLM:
    def __init__(self):
        self.max_task = 1
        self.keys : dict = self.load_keys(KEY_FILE)
        self.llms : list = self.init_llms()
        self.labelling_queue : list[tuple[int, str]] = self.get_labelling_queue()
        self.data : dict = self.get_data()
        self.tasks : list = self.separate_tasks(max_task=self.max_task)

    def update_queue(self) -> None:
        # Read the result file and use regex to extract all claims into a list
        # claim regex: "claim": "<claim_text>",
        claims = []
        if os.path.exists(OUTPUT_FILE):
            with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
                content = f.read()
                pattern = r'"claim":\s*"(.*?)"'
                claims = re.findall(pattern, content)

        # Remove all claims out from queue
        print(f"Removing {len(set(claims))} claims from the labelling queue...")
        self.labelling_queue = [
            item for item in self.labelling_queue if item[1] not in set(claims)
        ]
        self.tasks = self.separate_tasks(max_task=self.max_task)

    def load_keys(self, key_file) -> dict:
        with open(key_file, 'r') as f:
            keys = json.load(f)
        return keys

    def init_llms(self) -> list[PseudoGraphRelabellingLLM]:
        llms = []
        for key in self.keys.keys():
            llm = PseudoGraphRelabellingLLM(key=self.keys[key])
            llms.append(llm)
        return llms
    def get_labelling_queue(self) -> list:
        with open(QUEUE_FILE, 'rb') as f:
            queue = pickle.load(f)
        print(f"Loaded labelling queue with {len(queue)} items.")
        return queue

    def get_data(self) -> dict:
        with open(DATA_FILE, 'rb') as f:
            data = pickle.load(f)
        return data

    def separate_tasks(self, max_task: int = 249) -> list:
        """
        Separate each task to each LLM instance.
        """
        tasks = []
        for i in range(0, len(self.llms)):
            tasks.append(self.labelling_queue[i * max_task:(i + 1) * max_task])
        return tasks

    @staticmethod
    def extract_triplets(output: str) -> str:
        import re
        pattern = r"<e>.*?<\/e>\s*\|\|\s*.*?\s*\|\|\s*<e>.*?<\/e>"
        triplets = re.findall(pattern, output)
        return ';\n'.join(triplets)

    def append_result(self, claim: str, result: str, created_by: int) -> None:
        extracted_triplets = RelabellingLLM.extract_triplets(result)
        to_write = {
            "claim": claim,
            "triplets": extracted_triplets,
            "created_by": str(created_by)
        }
        with open(OUTPUT_FILE, 'a') as f:
            f.write(json.dumps(to_write) + '\n')


    def run_tasks(self, llm_index: int, task: list[tuple[int, str]]) -> None:
        """
        Run tasks for one LLM instance with a dedicated progress bar.
        """
        llm_instance : PseudoGraphRelabellingLLM = self.llms[llm_index]
        # Use position to stack progress bars vertically; leave=True keeps completed bars visible
        for item in tqdm(task, desc=f"LLM-{llm_index}", position=llm_index, leave=True):
            claim_id, claim = item
            response = llm_instance.submit(claim, self.data[claim])
            self.append_result(claim, response, llm_index)

    def run_all_tasks(self, max_workers: int = None) -> None:
        """
        Parallel run all tasks for all LLM instances using ThreadPoolExecutor.
        Each LLM instance gets its own stacked progress bar.

        Args:
            max_workers: Number of parallel workers (defaults to number of LLM instances)
        """
        if max_workers is None:
            max_workers = len(self.llms)

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            # Submit all tasks to the executor
            futures = {
                executor.submit(self.run_tasks, i, task): i
                for i, task in enumerate(self.tasks)
            }

            # Wait for all tasks to complete and handle exceptions
            for future in as_completed(futures):
                llm_index = futures[future]
                try:
                    future.result()
                    print(f"✓ LLM-{llm_index} completed all tasks")
                except Exception as e:
                    print(f"✗ LLM-{llm_index} failed with error: {e}")




r = RelabellingLLM()
r.update_queue()
r.run_all_tasks()  # runs all LLMs in parallel (default: one thread per LLM)

Loaded labelling queue with 1376 items.
Removing 1375 claims from the labelling queue...


LLM-2:   0%|          | 0/1 [00:00<?, ?it/s]

LLM-1:   0%|          | 0/1 [00:00<?, ?it/s]

LLM-3:   0%|          | 0/1 [00:00<?, ?it/s]

LLM-5:   0%|          | 0/1 [00:00<?, ?it/s]

LLM-4:   0%|          | 0/1 [00:00<?, ?it/s]

LLM-0:   0%|          | 0/1 [00:00<?, ?it/s]

✓ LLM-2 completed all tasks
✓ LLM-1 completed all tasks
✓ LLM-0 completed all tasks
✓ LLM-4 completed all tasks
✓ LLM-0 completed all tasks
✓ LLM-4 completed all tasks
✓ LLM-5 completed all tasks
✓ LLM-5 completed all tasks


✓ LLM-3 completed all tasks
